In [2]:
import pandas as pd

In [3]:
# 통합데이터 준비(merge : 합치기)
# products, orders, order_items
prod = pd.read_csv("../data/products.csv")
orders = pd.read_csv("../data/orders.csv")
oi = pd.read_csv("../data/order_items.csv")

In [4]:
# 중복된 상품 번호가 있는지 확인
# prod["product_id"].nunique()  # 497개
prod = prod.drop_duplicates("product_id")

In [5]:
# 카테고리 문제 확인: 공백 존재 => 삭제
prod["category"] = prod["category"].str.strip()

In [6]:
# orders dataframe 확인
orders.head()
# datetime 타입 변형(1. read_csv에서 parse_date. 2. to_datetime())
# 지금은 날짜 중요 x

,order_id,customer_id,order_datetime,channel,status
0,25463,1077,2024-06-25 00:17:41,store,delivered
1,171088,2998,2024-05-28 19:35:20,web,delivered
2,27437,4066,2024-02-14 17:49:14,app,delivered
3,98425,3243,2024-05-08 16:37:36,store,delivered
4,22325,5331,2024-04-17 20:08:22,app,delivered


In [14]:
# order_items : oi 데이터 정제
# order_item_id: 주문 상세 번호
# 1. 주문 상세 번호 중복 해결.
oi = oi.drop_duplicates("order_item_id")  # 행 단위 삭제 => 데이터프레임
# oi.info()
# 2. 단가: NaN 해결.
oi = oi.dropna(subset=["unit_price"])
# 3. 수량 < 0 해결.
oi = oi[oi["quantity"]>0]

# 전체 주문 금액.
oi["amount"] = oi["unit_price"] * oi["quantity"] * (1-oi["discount"])

In [16]:
oi.describe()

,order_item_id,order_id,product_id,quantity,unit_price,discount,amount
count,484287.000000,484287.000000,484287.000000,484287.000000,484287.000000,484287.000000,4.842870e+05
mean,249915.038033,99960.579425,220.309217,3.001212,50926.766566,0.249773,1.145918e+05
std,144300.108250,57753.899340,150.595851,1.412893,48910.628653,0.144480,1.371051e+05
min,1.000000,1.000000,1.000000,1.000000,1000.000000,0.000000,5.000000e+02
25%,124933.500000,49942.500000,92.000000,2.000000,15100.000000,0.120000,2.808600e+04
50%,249887.000000,99927.000000,216.000000,3.000000,27000.000000,0.250000,5.848000e+04
75%,374914.500000,149944.500000,348.000000,4.000000,73700.000000,0.370000,1.447500e+05
max,499880.000000,200000.000000,497.000000,5.000000,291000.000000,0.500000,1.440450e+06


In [19]:
# 1. orders, oi 조인 => 조인 조건: orders_id
m = oi.merge(orders, on="order_id")    # right(조인할 df or series), how(어떤 조인? 예. inner), on(조인할 컬럼명)이 핵심.
                                       # how는 inner가 default
m.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount,amount,customer_id,order_datetime,channel,status
0,59256,114625,3,1,20300.0,0.05,19285.0,5614,2024-06-13 16:58:05,app,delivered
1,230587,66337,80,2,90000.0,0.45,99000.0,5500,2024-04-02 18:18:21,app,paid
2,279813,163343,3,3,20300.0,0.23,46893.0,4823,2024-06-26 06:42:37,store,delivered
3,88487,180332,79,1,8600.0,0.38,5332.0,4266,2024-02-02 08:59:27,app,delivered
4,39240,174179,232,1,42600.0,0.31,29394.0,5564,2024-06-22 06:41:40,web,delivered


In [22]:
m = m.merge(prod, on="product_id")  # prod와 조인(merge)

m.head()        # 여기까지가 데이터 전처리.

,order_item_id,order_id,product_id,quantity,unit_price,discount,amount,customer_id,order_datetime,channel,status,product_name,category,price,cost
0,59256,114625,3,1,20300.0,0.05,19285.0,5614,2024-06-13 16:58:05,app,delivered,플러스 잡지,도서,20300.0,10500.0
1,230587,66337,80,2,90000.0,0.45,99000.0,5500,2024-04-02 18:18:21,app,paid,플러스 선반,가구,90000.0,65700.0
2,279813,163343,3,3,20300.0,0.23,46893.0,4823,2024-06-26 06:42:37,store,delivered,플러스 잡지,도서,20300.0,10500.0
3,88487,180332,79,1,8600.0,0.38,5332.0,4266,2024-02-02 08:59:27,app,delivered,플러스 립스틱,뷰티,8600.0,5900.0
4,39240,174179,232,1,42600.0,0.31,29394.0,5564,2024-06-22 06:41:40,web,delivered,프리미엄 립스틱,뷰티,42600.0,31400.0


In [ ]:
# cat_sales = (m.groupby("category")     # 여기까지는 grouping한 DataFrame 나옴

# m["category"].unique()  # ['도서', '가구', '뷰티', '전자', '의류', '식품']

l = m.groupby("category")["amount"].sum().sort_values(ascending=False)

category
전자    2.370663e+10
가구    1.688345e+10
의류    4.773476e+09
도서    4.550701e+09
뷰티    3.627218e+09
식품    1.953854e+09
Name: amount, dtype: float64

In [ ]:
# 카테고리별 매출, 건수, 평균단가
# m.groupby("category")   # 카테고리로 grouping: DataFrameGroupBy 리턴 => agg() : 집계(합, 개수, 평균)
report = m.groupby("category").agg(                  # 새로운 데이터프레임을 만듦.(agg)  
    sales = ("amount", "sum"),              # 카테고리별 매출
    orders = ("order_item_id", "count"),    # 카테고리별 개수
    avg_price = ("unit_price", "mean")      # 카테고리별 평균 단가
)

In [37]:
report

,sales,orders,avg_price
category,,,
가구,1.688345e+10,59089,127242.158439
도서,4.550701e+09,133445,15150.216194
뷰티,3.627218e+09,62009,26023.330484
식품,1.953854e+09,62371,13916.028282
의류,4.773476e+09,58637,36082.214302
전자,2.370663e+10,108736,96798.193790


In [40]:
m["cate_total"] = m.groupby("category")["amount"].transform("sum")
m["share"] = m["amount"] / m["cate_total"]
m[["category","amount","cate_total","share"]].head(10)

,category,amount,cate_total,share
0,도서,19285.0,4.550701e+09,0.000004
1,가구,99000.0,1.688345e+10,0.000006
2,도서,46893.0,4.550701e+09,0.000010
3,뷰티,5332.0,3.627218e+09,0.000001
4,뷰티,29394.0,3.627218e+09,0.000008
5,도서,11682.0,4.550701e+09,0.000003
6,도서,22736.0,4.550701e+09,0.000005
7,도서,19380.0,4.550701e+09,0.000004
8,도서,46575.0,4.550701e+09,0.000010
9,전자,308016.0,2.370663e+10,0.000013


In [45]:
# orders : 주문 데이터
# 채널, 상태별 개수
# orders["channel"].value_counts()
    # channel
    # web      109958
    # app       59894
    # store     30148
# orders["status"].value_counts()
    # status
    # delivered    109976
    # shipped       30017
    # paid          29993
    # canceled      20039
    # returned       9975

In [47]:
# 채널, 상태별 개수
combo = orders.groupby(["channel", "status"]).size()
combo

channel  status   
app      canceled      5865
         delivered    32845
         paid          9169
         returned      2975
         shipped       9040
store    canceled      3002
         delivered    16676
         paid          4460
         returned      1570
         shipped       4440
web      canceled     11172
         delivered    60455
         paid         16364
         returned      5430
         shipped      16537
dtype: int64

In [ ]:
type(combo)     # pandas.Series.  channel & status: multi index

pandas.Series

In [ ]:
# 카테고리, 주문 경로별 매출의 합.